# BiVLA on Google Colab

This notebook sets up the full BiVLA evaluation pipeline inside Google Colab and runs the three policy variants used in this repository:

- `openvla`
- `openvla_foveated`
- `openvla_retina`

It covers environment setup, package installation, headless rendering, optional Hugging Face authentication, single-task smoke tests, and the full evaluation matrix across all configured tasks.

Recommended runtime:

- Colab GPU runtime
- ideally an L4 or A100 if you want the exact repository path with `openvla/openvla-7b`
- high-RAM session if available
- enough disk space for the OpenVLA checkpoint and result files

Why that matters: the repository loads OpenVLA directly through `AutoModelForVision2Seq` and uses `bfloat16` on CUDA. Older Colab GPUs such as T4 can be memory-constrained for the exact setup and may not be reliable for a full run.


## Before you run it

This notebook assumes the repository is either already present on disk or can be cloned into Colab. If your OpenVLA checkpoint requires authentication, add `HF_TOKEN` as a Colab secret or environment variable before the authentication cell.

The full benchmark is expensive. The smoke-test cell is the right first run.


In [ ]:
# Basic configuration. Edit these values before running the setup cells.

REPO_DIR = "/content/BiVLA"
REPO_URL = ""  # Optional. Fill this if you want the notebook to clone the repo for you.
REPO_BRANCH = ""  # Optional. Leave empty to use the default branch.

OPENVLA_MODEL_PATH = "openvla/openvla-7b"
OPENVLA_UNNORM_KEY = "bridge_orig"
DEVICE = "cuda"

RESULTS_ROOT = f"{REPO_DIR}/results_colab"
DEFAULT_N_EPISODES = 24
SAVE_VIDEO = False


In [ ]:
import os
import pathlib
import shlex
import subprocess
import sys


def run_shell(cmd, cwd=None):
    printable = cmd if isinstance(cmd, str) else shlex.join(cmd)
    print("+", printable)
    subprocess.run(cmd, cwd=cwd, check=True, text=True)


repo_dir = pathlib.Path(REPO_DIR)
if (repo_dir / "simple_eval.py").exists():
    print(f"Using existing repo at {repo_dir}")
else:
    if not REPO_URL.strip():
        raise FileNotFoundError(
            "Repo not found at REPO_DIR. Either upload or mount the repository there, or set REPO_URL first."
        )
    clone_cmd = ["git", "clone"]
    if REPO_BRANCH.strip():
        clone_cmd += ["--branch", REPO_BRANCH.strip()]
    clone_cmd += [REPO_URL.strip(), REPO_DIR]
    run_shell(clone_cmd)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
print("Python:", sys.version.split()[0])


In [ ]:
%%bash
set -euxo pipefail
export DEBIAN_FRONTEND=noninteractive

apt-get update
apt-get install -y --no-install-recommends \
  build-essential \
  cmake \
  ffmpeg \
  git \
  libegl1 \
  libgl1 \
  libglib2.0-0 \
  libglfw3 \
  libjpeg-dev \
  libosmesa6 \
  libosmesa6-dev \
  libpng-dev \
  libsm6 \
  libvulkan1 \
  libxext6 \
  libxrender1 \
  mesa-vulkan-drivers \
  patchelf \
  xvfb

rm -rf /var/lib/apt/lists/*


In [ ]:
%%bash
set -euxo pipefail

python -m pip install --upgrade pip setuptools wheel
python -m pip install --upgrade "numpy<2.0" scipy==1.12.0
python -m pip install --upgrade pillow opencv-python imageio imageio-ffmpeg
python -m pip install --upgrade gymnasium==0.29.1 sapien==2.2.2 mani-skill2==0.5.0
python -m pip install --upgrade GitPython gdown h5py pyyaml tabulate tqdm
python -m pip install --upgrade transforms3d trimesh rtree ruckig
python -m pip install --upgrade accelerate einops huggingface_hub pandas sentencepiece timm transformers
python -m pip install -e ./SimplerEnv/ManiSkill2_real2sim
python -m pip install -e ./SimplerEnv


In [ ]:
import atexit
import subprocess
import time

repo_dir = pathlib.Path(REPO_DIR).resolve()
os.chdir(repo_dir)

os.environ["SIMPLER_ENV_PATH"] = str(repo_dir / "SimplerEnv")
os.environ["RESULTS_DIR"] = str(repo_dir / "results")
os.environ["OPENVLA_MODEL_PATH"] = OPENVLA_MODEL_PATH
os.environ["OPENVLA_UNNORM_KEY"] = OPENVLA_UNNORM_KEY
os.environ["BIVLA_VK_ICD"] = str(repo_dir / "configs" / "nvidia_icd_egl.json")
os.environ["MS2_REAL2SIM_ASSET_DIR"] = str(
    repo_dir / "SimplerEnv" / "ManiSkill2_real2sim" / "data"
)
os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
os.environ["DISPLAY"] = ":99"

subprocess.run("pkill -f 'Xvfb :99' || true", shell=True, check=True)
xvfb = subprocess.Popen(
    ["Xvfb", os.environ["DISPLAY"], "-screen", "0", "1400x900x24"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
atexit.register(lambda: xvfb.terminate() if xvfb.poll() is None else None)
time.sleep(2)

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")

if hf_token:
    from huggingface_hub import login
    login(hf_token, add_to_git_credential=False)
    print("Hugging Face token loaded.")
else:
    print("No HF_TOKEN found. If model download fails, add HF_TOKEN and rerun this cell.")

print("SIMPLER_ENV_PATH =", os.environ["SIMPLER_ENV_PATH"])
print("MS2_REAL2SIM_ASSET_DIR =", os.environ["MS2_REAL2SIM_ASSET_DIR"])
print("DISPLAY =", os.environ["DISPLAY"])


In [ ]:
import gymnasium
import sapien
import torch
import transformers

import simple_eval

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    if torch.cuda.get_device_capability(0)[0] < 8:
        print(
            "Warning: this GPU is older than Ampere. The exact repo path loads OpenVLA in bfloat16 on CUDA, so T4-class runtimes can be unstable or run out of memory."
        )
print("Transformers:", transformers.__version__)
print("Gymnasium:", gymnasium.__version__)
print("SAPIEN:", sapien.__version__)
print("Tasks:", list(simple_eval.TASK_CONFIGS.keys()))


In [ ]:
import json
import pathlib
import shlex
import subprocess
import sys
import time

import pandas as pd

MODELS = ["openvla", "openvla_foveated", "openvla_retina"]
TASKS = list(simple_eval.TASK_CONFIGS.keys())


def _flatten_extra_args(extra_args):
    flat = []
    for key, value in (extra_args or {}).items():
        flag = str(key)
        if isinstance(value, bool):
            if value:
                flat.append(flag)
        elif isinstance(value, (list, tuple)):
            flat.extend([flag, ",".join(map(str, value))])
        else:
            flat.extend([flag, str(value)])
    return flat


def run_eval(
    model,
    task,
    n_episodes=DEFAULT_N_EPISODES,
    output_root=RESULTS_ROOT,
    save_video=SAVE_VIDEO,
    extra_args=None,
):
    output_dir = pathlib.Path(output_root) / model / task
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        str(repo_dir / "simple_eval.py"),
        "--model",
        model,
        "--task",
        task,
        "--n-episodes",
        str(n_episodes),
        "--output-dir",
        str(output_dir),
        "--openvla-model-path",
        OPENVLA_MODEL_PATH,
        "--openvla-unnorm-key",
        OPENVLA_UNNORM_KEY,
        "--device",
        DEVICE,
    ]
    if save_video:
        cmd.append("--save-video")
    cmd.extend(_flatten_extra_args(extra_args))

    print("+", shlex.join(cmd))
    start = time.time()
    subprocess.run(cmd, cwd=repo_dir, check=True)
    print(f"Finished in {time.time() - start:.1f}s")

    result_path = output_dir / f"results_{task}.json"
    with open(result_path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def summaries_to_frame(summaries):
    rows = []
    for item in summaries:
        rows.append(
            {
                "model": item["model"],
                "task": item["task"],
                "success_rate": item["success_rate"],
                "grasp_rate": item["grasp_rate"],
                "avg_steps": item["avg_steps"],
                "avg_elapsed": item["avg_elapsed"],
                "model_call_rate": (item.get("model_stats") or {}).get("model_call_rate"),
                "speedup_vs_vanilla_est": (item.get("model_stats") or {}).get(
                    "speedup_vs_vanilla_est"
                ),
            }
        )
    return pd.DataFrame(rows).sort_values(["task", "model"]).reset_index(drop=True)


## Smoke test

Run a minimal check before launching the full benchmark. This executes all three model variants on one task with a very small episode count.


In [ ]:
SMOKE_TASK = "widowx_spoon_on_towel"
SMOKE_EPISODES = 1

smoke_summaries = []
for model in MODELS:
    smoke_summaries.append(run_eval(model, SMOKE_TASK, n_episodes=SMOKE_EPISODES))

summaries_to_frame(smoke_summaries)


## Full run across all three versions

This cell runs the complete model-task matrix defined in the repository. By default that means 3 policy variants x 4 tasks. If you want a shorter pass, reduce `N_EPISODES` or trim `RUN_MODELS` and `RUN_TASKS` first.


In [ ]:
N_EPISODES = DEFAULT_N_EPISODES
RUN_MODELS = MODELS
RUN_TASKS = TASKS

all_summaries = []
for task in RUN_TASKS:
    for model in RUN_MODELS:
        all_summaries.append(run_eval(model, task, n_episodes=N_EPISODES))

results_table = summaries_to_frame(all_summaries)
results_table


In [ ]:
summary_path = pathlib.Path(RESULTS_ROOT) / "colab_summary.csv"
results_table.to_csv(summary_path, index=False)
print("Saved summary table to", summary_path)
results_table
